In [13]:
import pandas as pd
import numpy as np
import kagglehub
from kagglehub import KaggleDatasetAdapter

In [ ]:
# # Download latest version
# path = kagglehub.dataset_download("vivek468/superstore-dataset-final")

# print("Path to dataset files:", path)

100%|██████████| 550k/550k [00:01<00:00, 452kB/s]

Extracting files...
Path to dataset files: C:\Users\Furqan\.cache\kagglehub\datasets\vivek468\superstore-dataset-final\versions\1


In [9]:
df = pd.read_csv("Superstore.csv", encoding='latin1')

Q1. Sales per Order ID + Region distribution

The dataset was loaded and a Series representing total sales per Order ID was created using groupby. Descriptive statistics were computed for each region to analyze the distribution of sales. Index-based selection was used to examine specific regional performance. This helps in understanding regional sales variability and central tendencies.

In [20]:

sales_per_order = df.groupby("Order ID")["Sales"].sum()

print(sales_per_order.head())


region_stats = df.groupby("Region")["Sales"].describe()
print(region_stats)


print(region_stats.loc["West"])   # example

Order ID
CA-2014-100006    377.970
CA-2014-100090    699.192
CA-2014-100293     91.056
CA-2014-100328      3.928
CA-2014-100363     21.376
Name: Sales, dtype: float64
          count        mean         std    min     25%     50%       75%  \
Region                                                                     
Central  2323.0  215.772661  632.779010  0.444  14.620  45.980  200.0120   
East     2848.0  238.336110  620.712652  0.852  17.520  54.900  209.6175   
South    1620.0  241.803645  774.796273  1.167  17.187  54.594  208.7220   
West     3203.0  226.493233  524.876877  0.990  19.440  60.840  215.8090   

               max  
Region              
Central  17499.950  
East     11199.968  
South    22638.480  
West     13999.960  
count     3203.000000
mean       226.493233
std        524.876877
min          0.990000
25%         19.440000
50%         60.840000
75%        215.809000
max      13999.960000
Name: West, dtype: float64


Q2. Highest Profit Margin Category/Sub-category

Profit margin was calculated as the ratio of total profit to total sales for each category and sub-category. GroupBy aggregation was used to compute these values efficiently. The analysis identifies the most profitable segments as well as underperforming ones with low or negative profit margins, enabling better business decision-making.

In [ ]:

df["Profit_Margin"] = df["Profit"] / df["Sales"]

grouped = df.groupby(["Category", "Sub-Category"]).agg({
    "Sales": "sum",
    "Profit": "sum"
})

grouped["Profit_Margin"] = grouped["Profit"] / grouped["Sales"]


print(grouped.sort_values("Profit_Margin", ascending=False).head())


print(grouped[grouped["Profit_Margin"] < 0])

                                   Sales      Profit  Profit_Margin
Category        Sub-Category                                       
Office Supplies Labels         12486.312   5546.2540       0.444187
                Paper          78479.206  34053.5693       0.433918
                Envelopes      16476.402   6964.1767       0.422676
Technology      Copiers       149528.030  55617.8249       0.371956
Office Supplies Fasteners       3024.280    949.5182       0.313965
                                    Sales      Profit  Profit_Margin
Category        Sub-Category                                        
Furniture       Bookcases     114879.9963  -3472.5560      -0.030228
                Tables        206965.5320 -17725.4811      -0.085645
Office Supplies Supplies       46673.5380  -1189.0995      -0.025477


Q3. Hierarchical Index (Region, Category)

A hierarchical index was created using Region and Category to enable multi-level analysis. Aggregations were performed to compute total sales and profit at each level. Cross-section (xs) was used to extract region-specific insights. This structure helps identify strong and weak regions across product categories.

In [ ]:
multi = df.set_index(["Region", "Category"])


agg_data = multi.groupby(level=[0,1])[["Sales", "Profit"]].sum()
print(agg_data)


print(agg_data.xs("West", level="Region"))

                               Sales      Profit
Region  Category                                
Central Furniture        163797.1638  -2871.0494
        Office Supplies  167026.4150   8879.9799
        Technology       170416.3120  33697.4320
East    Furniture        208291.2040   3046.1658
        Office Supplies  205516.0550  41014.5791
        Technology       264973.9810  47462.0351
South   Furniture        117298.6840   6771.2061
        Office Supplies  125651.3130  19986.3928
        Technology       148771.9080  19991.8314
West    Furniture        252612.7435  11504.9503
        Office Supplies  220853.2490  52609.8490
        Technology       251991.8320  44303.6496
                       Sales      Profit
Category                                
Furniture        252612.7435  11504.9503
Office Supplies  220853.2490  52609.8490
Technology       251991.8320  44303.6496


Q4. Profitability Score (ufunc-based)

A composite Profitability Score was developed using normalized values of Profit, Discount, and Quantity. NumPy-based operations were used for normalization and calculation. Products were ranked based on this score to identify top performers. This provides a holistic evaluation of product performance.

In [ ]:


df["Profit_norm"] = (df["Profit"] - df["Profit"].min()) / (df["Profit"].max() - df["Profit"].min())
df["Discount_norm"] = (df["Discount"] - df["Discount"].min()) / (df["Discount"].max() - df["Discount"].min())
df["Quantity_norm"] = (df["Quantity"] - df["Quantity"].min()) / (df["Quantity"].max() - df["Quantity"].min())


df["Score"] = df["Profit_norm"] - df["Discount_norm"] + df["Quantity_norm"]


top_products = df.sort_values("Score", ascending=False)[["Product Name", "Score"]].head(10)
print(top_products)

                                           Product Name     Score
9039   GBC Ibimaster 500 Manual ProClick Binding System  1.692836
1711  Acco 7-Outlet Masterpiece Power Center, Wihtou...  1.474042
7387  Electrix Architect's Clamp-On Swing Arm Lamp, ...  1.465838
1429  Avery 4027 File Folder Labels for Dot Matrix P...  1.453107
1433                  High-Back Leather Manager's Chair  1.450919
9168                                         Xerox 1964  1.449806
6628                  PureGear Roll-On Screen Protector  1.448955
9941  Memorex Mini Travel Drive 16 GB USB 2.0 Flash ...  1.445813
575         Personal Creations Ink Jet Cards and Labels  1.445250
3902  Eldon ProFile File 'N Store Portable File Tub ...  1.445046


Q5. Null Values Impact

Null values in the dataset were identified and analyzed. Two approaches were compared: removing null rows and imputing missing values using group-level means. The impact of both methods on total revenue was examined, highlighting how data cleaning strategies affect overall analysis.

In [ ]:

print(df.isnull().sum())


drop_df = df.dropna()
drop_revenue = drop_df["Sales"].sum()


impute_df = df.copy()
impute_df["Sales"] = impute_df.groupby("Region")["Sales"].transform(lambda x: x.fillna(x.mean()))
impute_revenue = impute_df["Sales"].sum()

print("Original:", df["Sales"].sum())
print("Drop:", drop_revenue)
print("Imputed:", impute_revenue)

Row ID           0
Order ID         0
Order Date       0
Ship Date        0
Ship Mode        0
Customer ID      0
Customer Name    0
Segment          0
Country          0
City             0
State            0
Postal Code      0
Region           0
Product ID       0
Category         0
Sub-Category     0
Product Name     0
Sales            0
Quantity         0
Discount         0
Profit           0
Profit_Margin    0
Profit_norm      0
Discount_norm    0
Quantity_norm    0
Score            0
dtype: int64
Original: 2297200.8603
Drop: 2297200.8603
Imputed: 2297200.8603


Q6. Loss-making Discounts

Boolean indexing was used to filter orders with high discounts (above 30%) and negative profit. The filtered data was grouped by sub-category to identify loss-making segments. A comparison with overall averages was performed to evaluate the severity of losses under high discount conditions.

In [16]:
filtered = df[(df["Discount"] > 0.3) & (df["Profit"] < 0)]

loss_by_subcat = filtered.groupby("Sub-Category")["Profit"].mean()
overall_avg = df.groupby("Sub-Category")["Profit"].mean()

comparison = pd.DataFrame({
    "High_Discount_Loss": loss_by_subcat,
    "Overall": overall_avg
})

print(comparison.sort_values("High_Discount_Loss"))

              High_Discount_Loss     Overall
Sub-Category                                
Machines             -699.981353   29.432669
Tables               -223.736846  -55.565771
Bookcases            -175.698147  -15.230509
Appliances           -128.800615   38.922758
Phones                -69.234845   50.073938
Binders               -62.822996   19.843574
Furnishings           -43.077212   13.645918
Accessories                  NaN   54.111788
Art                          NaN    8.200737
Chairs                       NaN   43.095894
Copiers                      NaN  817.909190
Envelopes                    NaN   27.418019
Fasteners                    NaN    4.375660
Labels                       NaN   15.236962
Paper                        NaN   24.856620
Storage                      NaN   25.152277
Supplies                     NaN   -6.258418


Q7. Shipping Performance (apply)

A custom function was implemented using apply() to classify orders as 'On-Time' or 'Delayed' based on shipping duration. The results were aggregated region-wise to generate a delay summary report. This helps evaluate shipping efficiency across different regions.

In [ ]:
df["Order Date"] = pd.to_datetime(df["Order Date"])
df["Ship Date"] = pd.to_datetime(df["Ship Date"])

def shipping_status(row):
    return "Delayed" if (row["Ship Date"] - row["Order Date"]).days > 5 else "On-Time"

df["Shipping_Status"] = df.apply(shipping_status, axis=1)

summary = df.groupby("Region")["Shipping_Status"].value_counts().unstack()
print(summary)

Shipping_Status  Delayed  On-Time
Region                           
Central              443     1880
East                 496     2352
South                284     1336
West                 601     2602


Q8. Index Alignment Behavior

Region-wise Sales and Profit Series were combined to demonstrate index alignment behavior in Pandas. When indices did not match, NaN values were introduced automatically. This highlights how Pandas aligns data based on labels rather than position during operations.

In [ ]:
sales_series = df.groupby("Region")["Sales"].sum()
profit_series = df.groupby("Region")["Profit"].sum()

profit_series = profit_series.drop("South")

combined = sales_series + profit_series
print(combined)

Region
Central    540946.2533
East       770304.0200
South              NaN
West       833876.2734
dtype: float64


Q9. Hierarchical Index (Segment, Category)

Hierarchical indexing was applied to Segment and Category to compute average sales, profit, and quantity per group. The results were reshaped using unstack() to create a matrix format, improving readability and enabling easier comparison across segments and categories.

In [ ]:
multi2 = df.groupby(["Segment", "Category"]).agg({
    "Sales": "mean",
    "Profit": "mean",
    "Quantity": "mean"
})

matrix = multi2.unstack()
print(matrix)

                  Sales                                 Profit  \
Category      Furniture Office Supplies  Technology  Furniture   
Segment                                                          
Consumer     351.347091      116.390194  427.339534   6.281293   
Corporate    354.519792      126.745309  444.855810  11.741201   
Home Office  336.825131      115.309021  535.976658  10.705465   

                                        Quantity                             
Category    Office Supplies Technology Furniture Office Supplies Technology  
Segment                                                                      
Consumer          18.014174  74.445646  3.743037        3.760154   3.782334  
Corporate         22.102923  79.723823  3.862229        3.856044   3.781588  
Home Office       24.034439  89.152458  3.776243        3.827618   3.646199  


Q10. Final Statement and Conclusion

All the above analyses have been performed and compiled in a structured report format within this Jupyter Notebook, including data preprocessing, hierarchical analysis, KPI computation, and profit/loss evaluation.